In [9]:
import os

from google.cloud import vision
from google.cloud.vision_v1 import types

In [5]:
#the JSON file you downloaded in step 5 above
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = ''

SyntaxError: invalid syntax (3738503923.py, line 1)

In [1]:
import pandas as pd
import os
import cv2
import numpy as np
import json

import imutils
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

#Load index list
Year='1942'
Showa='17'


In [2]:
#Encoding Function
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)


In [3]:
### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

def Clova(Image):
    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(Image).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json    

In [4]:
#Function for Cell
def GetCell(cropped):
    #Code for Adding Grid
        ##Right page
        img = cropped.copy()
        hh, ww = img.shape[:2]

        #Identify grid location
        ## convert to grayscale
        gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
        # threshold gray image
        thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)[1]

        ## count number of non-zero pixels in each column and row. 
        countCol = np.count_nonzero(thresh, axis=0)
        countRow = np.count_nonzero(thresh, axis=1)

        ###############
        ## Column lines
        ###############
        ### This finds the height of the smallest peak
        peaksCol, _ = find_peaks(countCol, distance=10)
        ### threshold count at Thres
        count_thresh = countCol.copy()
        count_thresh[peaksCol] = 255
        count_thresh[count_thresh!=255] = 0
        count_thresh = count_thresh.astype(np.uint8)

        ### get contours
        contoursCol = cv2.findContours(count_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contoursCol = contoursCol[0] if len(contoursCol) == 2 else contoursCol[1]

        ### loop over contours and get bounding boxes and ycenter and draw horizontal line at ycenter
        result = cropped.copy()
        for cntr in contoursCol:
            x,y,w,h = cv2.boundingRect(cntr)
            ycenter = y
            cv2.line(result, (ycenter,0), (ycenter,hh), (255, 0, 0), 1)
        

        ###############
        ## Row lines
        ###############
        peaksRow, _ = find_peaks(countRow, distance=60)
        ### threshold count at Thres
        count_thresh = countRow.copy()
        count_thresh[peaksRow] = 255
        count_thresh[count_thresh!=255] = 0
        count_thresh = count_thresh.astype(np.uint8)

        ### get contours
        contoursRow = cv2.findContours(count_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contoursRow = contoursRow[0] if len(contoursRow) == 2 else contoursRow[1]

        ### loop over contours and get bounding boxes and ycenter and draw horizontal line at ycenter
        for cntr in contoursRow:
            x,y,w,h = cv2.boundingRect(cntr)
            ycenter = y+h//2
            cv2.line(result, (0,ycenter), (hh,ycenter), (255, 0, 0), 1)
                
        return(peaksRow,peaksCol)

In [5]:
def Extract(Position,ImageNumber):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Position+"\\"
    stream = open(path+"Image"+str(ImageNumber)+'.png', "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

    HH,WW=img.shape[:2]
    
    dfA = pd.DataFrame(columns = ['Name', 'Wage'])
    dfT = pd.DataFrame(columns = ['Name', 'Wage'])
    dfM = pd.DataFrame(columns = ['Name', 'Wage'])
    dfB = pd.DataFrame(columns = ['Name', 'Wage'])
    
    Part=dta[Year][Dept][Office]['Positions'][Position]['Part']
    
    if Part==1:
        cropped=img
        HH,WW=cropped.shape[:2]
        MiddleLineList=GetCell(cropped)[0]
        res = list(map(abs, [d-HH//2 for d in MiddleLineList.tolist()]))
        minpos = res.index(min(res))
        MiddleLine=200
        ColumnLine=GetCell(cropped)[1].tolist()
        ColumnLine=[d for d in ColumnLine if d>15]
        
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]={}
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['Col']=ColumnLine
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['MidRow']=MiddleLine
        
        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
        for Line in ColumnLine:
            if Line==ColumnLine[0]:
                #Wage
                Image=cropped[0:MiddleLine,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfA = dfA.append(df2, ignore_index = True)

            else:
                #Wage
                Image=cropped[0:MiddleLine,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])


                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfA = dfA.append(df2, ignore_index = True)
        return(dfA)
    
    if Part==2:
        cropped=img[0:HH//2,0:WW]
        MiddleLineList=GetCell(cropped)[0]
        res = list(map(abs, [d-HH//4 for d in MiddleLineList.tolist()]))
        minpos = res.index(min(res))

        MiddleLine=MiddleLineList[minpos]
        ColumnLine=GetCell(cropped)[1]
        ColumnLine=[d for d in ColumnLine if d>15]
        
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]={}
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['TopCol']=ColumnLine
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['TopMid']=MiddleLine

        
        ##Top Chunk ##
        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
        HH,WW=cropped.shape[:2]
        for Line in ColumnLine:
            if Line==ColumnLine[0]:
                #Wage
                Image=cropped[0:MiddleLine,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfT = dfT.append(df2, ignore_index = True)
            else:
                #Wage
                Image=cropped[0:MiddleLine,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfT = dfT.append(df2, ignore_index = True)

        ##Bottom Chunk##
        cropped=img[HH//2:HH,0:WW]
        HH,WW=cropped.shape[:2]
        MiddleLineList=GetCell(cropped)[0]
        res = list(map(abs, [d-HH//4 for d in MiddleLineList.tolist()]))
        minpos = res.index(min(res))
        MiddleLine=MiddleLineList[minpos]
        ColumnLine=GetCell(cropped)[1]
        ColumnLine=[d for d in ColumnLine if d>15]
                
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['BtmCol']=ColumnLine
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['BtmMid']=MiddleLine

        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
        for Line in ColumnLine:
            if Line==ColumnLine[0]:
                #Wage
                Image=cropped[0:MiddleLine,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfB = dfB.append(df2, ignore_index = True)
            else:
                #Wage
                Image=cropped[0:MiddleLine,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Wage='NA'
                else:
                    Wage=''.join([d['inferText'] for d in Json['fields']])

                #Name
                Image=cropped[MiddleLine:HH,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfB = dfB.append(df2, ignore_index = True)
        return pd.concat([dfT,dfB], ignore_index = True)
    
    if Part==3:
         ##Top Chunk ##
        cropped=img[0:HH//3,0:WW]
        MiddleLineList=GetCell(cropped)[0]
        res = list(map(abs, [d-HH//4 for d in MiddleLineList.tolist()]))
        minpos = res.index(min(res))

        MiddleLine=MiddleLineList[minpos]
        ColumnLine=GetCell(cropped)[1]
        ColumnLine=[d for d in ColumnLine if d>15]
        
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]={}
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['TopCol']=ColumnLine
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['TopMid']=MiddleLine        
        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
        HH,WW=cropped.shape[:2]
        for Line in ColumnLine:
            if Line==ColumnLine[0]:
                #Wage
                Wage='NA'

                #Name
                Image=cropped[0:HH,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfT = dfT.append(df2, ignore_index = True)
            else:
                #Wage
                Wage='NA'
                
                #Name
                Image=cropped[MiddleLine:HH,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfT = dfT.append(df2, ignore_index = True)
                
        ##Middle Chunk##
        cropped=img[HH//3:2*HH//3,0:WW]
        HH,WW=cropped.shape[:2]
        MiddleLineList=GetCell(cropped)[0]
        res = list(map(abs, [d-HH//4 for d in MiddleLineList.tolist()]))
        minpos = res.index(min(res))
        MiddleLine=MiddleLineList[minpos]
        ColumnLine=GetCell(cropped)[1]
        ColumnLine=[d for d in ColumnLine if d>15]
                
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['BtmCol']=ColumnLine
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['BtmMid']=MiddleLine

        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
        for Line in ColumnLine:
            if Line==ColumnLine[0]:
                #Wage
                Wage='NA'

                #Name
                Image=cropped[MiddleLine:HH,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfM = dfM.append(df2, ignore_index = True)
            else:
                #Wage
                Wage='NA'
                
                #Name
                Image=cropped[MiddleLine:HH,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfM = dfM.append(df2, ignore_index = True)

        ##Bottom Chunk##
        cropped=img[2*HH//3:HH,0:WW]
        HH,WW=cropped.shape[:2]
        MiddleLineList=GetCell(cropped)[0]
        res = list(map(abs, [d-HH//4 for d in MiddleLineList.tolist()]))
        minpos = res.index(min(res))
        MiddleLine=MiddleLineList[minpos]
        ColumnLine=GetCell(cropped)[1]
        ColumnLine=[d for d in ColumnLine if d>15]
                
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['BtmCol']=ColumnLine
        dta[Year][Dept][Office]['Positions'][Position]['Image'+str(ImageNumber)]['BtmMid']=MiddleLine

        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
        for Line in ColumnLine:
            if Line==ColumnLine[0]:
                #Wage
                Wage='NA'
                
                #Name
                Image=cropped[MiddleLine:HH,0:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfB = dfB.append(df2, ignore_index = True)
            else:
                #Wage
                Wage='NA'
                
                #Name
                Image=cropped[MiddleLine:HH,Line-15:Line]
                cv2.imwrite(path+"Temp.jpg",Image)
                with open(path+"Temp.jpg",'rb') as f:
                    file_data = f.read()
                Json=Clova(file_data)
                if Json['inferResult']=='ERROR':
                    Name='NA'
                else:
                    Name=''.join([d['inferText'] for d in Json['fields']])

                #Add to DF
                #Add to DF
                df2 = {'Name': Name, 'Wage': Wage}
                dfB = dfB.append(df2, ignore_index = True)
        return pd.concat([dfT,dfM,dfB], ignore_index = True)

In [6]:
def Check(df,n):
    Row  = df.iloc[n]
    Dept=Row["Dept"]
    Office=Row["Office"]
    Unit=Row['Unit']
    Posi=Row['Position']
    
    print('['+str(n)+',"'+Dept+'","'+Office+'","'+Unit+'","'+Posi+'"]')
    
    StrPage=int(dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['Starting_Page'])
    EndPage=int(dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['EndPage'])
    
    PageRange=range(1,EndPage-StrPage+2)
    if (Posi=='Admin') or (Posi=='Clerk1'):
        for ImageNumber in PageRange:
            path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Unit+"\\"+Position+"\\"
            stream = open(path+"Image"+str(ImageNumber)+'.png', "rb")
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            HH,WW=img.shape[:2]
            
            Topimg=img[0:HH//2,0:WW]
            Mid=dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['Image'+str(ImageNumber)]['TopMid']
            Col=dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['Image'+str(ImageNumber)]['TopCol']
            
            cv2.line(Topimg,(0,Mid),(WW,Mid),(225,0,0),2)
            
            for Line in Col:
                cv2.line(Topimg,(Line,0),(Line,HH//2),(225,0,0),2)
            cv2.imshow(str(ImageNumber),Topimg)
            cv2.waitKey(0)
            
            Btmimg=img[HH//2:HH,0:WW]
            Mid=dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['Image'+str(ImageNumber)]['BtmMid']
            Col=dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['Image'+str(ImageNumber)]['BtmCol']
            
            cv2.line(Btmimg,(0,Mid),(WW,Mid),(225,0,0),2)
            
            for Line in Col:
                cv2.line(Btmimg,(Line,0),(Line,HH//2),(225,0,0),2)
            cv2.imshow(str(ImageNumber),Btmimg)
            cv2.waitKey(0)
    else:
        for ImageNumber in PageRange:
            path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Unit+"\\"+Position+"\\"
            stream = open(path+"Image"+str(ImageNumber)+'.png', "rb")
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
            HH,WW=img.shape[:2]
            
            Mid=dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['Image'+str(ImageNumber)]['MidRow']
            Col=dta[Year][Dept][Office]['Units'][Unit]['Positions'][Posi]['Image'+str(ImageNumber)]['Col']
            
            cv2.line(img,(0,Mid),(WW,Mid),(225,0,0),2)
            
            for Line in Col:
                cv2.line(img,(Line,0),(Line,HH),(225,0,0),2)
            cv2.imshow(str(ImageNumber),img)
            cv2.waitKey(0)

In [7]:
#Load Data Frame
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\"+str(Year)+"\\DataFrame_PositionFin.json"
with open(path, 'r') as j:
     dta = json.loads(j.read())

df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+str(Showa)+'.csv')
df=df.drop(df.columns[0], axis=1)

In [8]:
#Extract key info of office
n=3
Row  = df.iloc[n]
print(Row)
###Collect Location information###
Dept=Row["Dept"]
Office=Row["Office"]
PositionList=list(dta[str(Year)][Dept][Office]['Positions'].keys())
print(PositionList)

for Position in PositionList:
    StrPage=int(dta[str(Year)][Dept][Office]['Positions'][Position]['StrPage'])
    EndPage=int(dta[str(Year)][Dept][Office]['Positions'][Position]['EndPage'])
    PageList=list(set([1,EndPage-StrPage+1]))
    print(Position)
    for ImageNumber in PageList:        
        print('Image Number is '+str(ImageNumber))
        #Download Image
        path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Position+"\\"
        try:
            stream = open(path+"Image"+str(ImageNumber)+'.png', "rb")
            bytes = bytearray(stream.read())
            numpyarray = np.asarray(bytes, dtype=np.uint8)
            img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
        except:
            print('Could not find image')
            continue

        HH,WW=img.shape[:2]

        DF=pd.DataFrame(columns = ['Name', 'Wage'])
        dta[str(Year)][Dept][Office]['Positions'][Position]['Data']=Extract(Position,ImageNumber)

Office    吏務課
Dept      総務部
Year       17
Page        5
Name: 3, dtype: object
['Manager', 'Clerk1', 'Worker', 'Outsource']
Manager
Image Number is 1


C:\Temp\ipykernel_23884\4122007360.py:58: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from 

Image Number is 2


C:\Temp\ipykernel_23884\4122007360.py:58: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from 

Clerk1
Image Number is 1


...

C:\Temp\ipykernel_23884\4122007360.py:265: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:283: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:283: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:283: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfT = dfT.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:283: FutureWarning: The frame.append method is deprecated and will be removed 

Outsource
Image Number is 1


C:\Temp\ipykernel_23884\4122007360.py:58: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)


Image Number is 5
Could not find image


C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)


In [12]:
n=3
Row  = df.iloc[n]
print(Row)
###Collect Location information###
Dept=Row["Dept"]
Office=Row["Office"]
PositionList=list(dta[str(Year)][Dept][Office]['Positions'].keys())
print(PositionList)
dta[str(Year)][Dept][Office]['Positions']

Office    吏務課
Dept      総務部
Year       17
Page        5
Name: 3, dtype: object
['Manager', 'Clerk1', 'Worker', 'Outsource']


{'Manager': {'StrPage': 1.0,
  'StrLocation': 200.0,
  'Part': 1.0,
  'EndLocation': 374.0,
  'EndPage': 2.0,
  'Image1': {'Col': [69, 90, 109, 130, 142, 152, 168, 183, 196],
   'MidRow': 200},
  'Data':            Name    Wage
  0  制長長1下#浩=/郎主事       :
  1         本、副秋刀        
  2           房使義  大事定員講習
  3            は迂        
  4       成長長汾光更具      卆拉
  5                      
  6                      ,
  'Image2': {'Col': [24, 37, 47, 58, 75, 100, 120], 'MidRow': 200}},
 'Clerk1': {'StrPage': 2.0,
  'StrLocation': 374.0,
  'Part': 2.0,
  'EndLocation': 140.0,
  'EndPage': 2.0,
  'Image1': {'TopCol': [35,
    49,
    65,
    82,
    98,
    110,
    125,
    142,
    161,
    174,
    190,
    205,
    216,
    236,
    246,
    263,
    278,
    292,
    308,
    321,
    336,
    352,
    368],
   'TopMid': 52,
   'BtmCol': [21,
    35,
    49,
    67,
    80,
    98,
    113,
    125,
    135,
    146,
    158,
    173,
    190,
    202,
    216,
    233,
    246,
    263,
    2

In [15]:
#Test code| Version 2#
#Show Working office#
FailedList=[]
for n in range(20,len(df)):
    #Extract key info of office
    Row  = df.iloc[n]
    print(Row)
    ###Collect Location information###
    Dept=Row["Dept"]
    Office=Row["Office"]
    try:
        PositionList=list(dta[str(Year)][Dept][Office]['Positions'].keys())
        print(PositionList)
    except:
        FailedList.append([Dept,Office,Position,'All'])
        continue

    for Position in PositionList:
        try:            
            StrPage=int(dta[str(Year)][Dept][Office]['Positions'][Position]['StrPage'])
            EndPage=int(dta[str(Year)][Dept][Office]['Positions'][Position]['EndPage'])
            PageList=list(set([1,EndPage-StrPage+1]))
            print(Position)
        except:
            FailedList.append([Dept,Office,Position,'All'])
            continue
        for ImageNumber in PageList:
            try:
                print('Image Number is '+str(ImageNumber))
                #Download Image
                path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Office_Level\\"+Year+"\\"+Dept+"\\"+Office+"\\"+Position+"\\"
                stream = open(path+"Image"+str(ImageNumber)+'.png', "rb")
                bytes = bytearray(stream.read())
                numpyarray = np.asarray(bytes, dtype=np.uint8)
                img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)

                HH,WW=img.shape[:2]

                DF=pd.DataFrame(columns = ['Name', 'Wage'])
                dta[str(Year)][Dept][Office]['Positions'][Position]['Data']=Extract(Position,ImageNumber)
            except:
                FailedList.append([Dept,Office,Position,ImageNumber])
                continue

Office    学校営業課
Dept        建築部
Year         17
Page         21
Name: 20, dtype: object
['Engineer1', 'Clerk1', 'Engineer2', 'Worker', 'Outsource']
Engineer1
Image Number is 1


C:\Temp\ipykernel_23884\4122007360.py:58: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from 

Image Number is 3
Clerk1
Image Number is 1


C:\Temp\ipykernel_23884\4122007360.py:58: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from 

Engineer2
Image Number is 1
Image Number is 2
Worker
Image Number is 1


...

C:\Temp\ipykernel_23884\4122007360.py:58: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from 

Engineer2
Image Number is 1
Worker
Image Number is 1


C:\Temp\ipykernel_23884\4122007360.py:58: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)
C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from 

Engineer1
Image Number is 136
Image Number is 1
Office     病院
Dept      事業部
Year       17
Page      153
Name: 100, dtype: object
['Manager', 'Engineer2', 'Worker']
Manager
Image Number is 1
Engineer2
Image Number is 1
Image Number is 3
Worker
Image Number is 1
Office      庶務課
Dept      電気研究所
Year         17
Page        155
Name: 101, dtype: object
Office      試験課
Dept      電気研究所
Year         17
Page        155
Name: 102, dtype: object
['Engineer1', 'Manager', 'Engineer2', 'Clerk1']
Engineer1
Image Number is 1
Manager
Image Number is 1
Engineer2
Image Number is 1
Image Number is 2
Clerk1
Image Number is 1
Office      研究課
Dept      電気研究所
Year         17
Page        156
Name: 103, dtype: object
['Leader', 'Engineer1', 'Engineer2']
Leader
Image Number is 1
Engineer1
Image Number is 1
Engineer2
Image Number is 1
Office    管理課
Dept      清掃部
Year       17
Page      156
Name: 104, dtype: object
['Manager', 'Clerk1', 'Monitor']
Manager
Image Number is 1
Clerk1
Image Number is 1
Image Number is 

C:\Temp\ipykernel_23884\4122007360.py:86: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  dfA = dfA.append(df2, ignore_index = True)


In [16]:
n

106

In [17]:
Frame=pd.DataFrame(columns=['Dept', 'Office','Position','Name','Wage'])
FailedList=[]
for n in range(1,12):
    Row=df.iloc[n]
    Dept=Row['Dept']
    Office=Row['Office']
    try:
        PosiList=list(dta[Year][Dept][Office]['Positions'].keys())
    except:
        FailedList.append(list((Dept,Office)))
        continue
        
    for Posi in PosiList:
        NewFrame=dta[Year][Dept][Office]['Positions'][Posi]['Data']
        NewFrame['Dept']=Dept
        NewFrame['Office']=Office
        NewFrame['Positions']=Posi
        Frame=pd.concat([Frame,NewFrame])
Frame.to_csv("C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\"+Year+"\\Final.csv")

In [ ]:
for n in range(3,len(df)+1):
    Check(df,n)
    
cv2.destroyAllWindows()
cv2.waitKey(0)

,Dept,Office,Position,Name,Wage,Positions
9,企書部,都市計書課,NaN,木郎,青九五,Engineer2
10,企書部,都市計書課,NaN,大金之助月九五山,大,Engineer2
11,企書部,都市計書課,NaN,,,Engineer2
12,企書部,都市計書課,NaN,芳賀和,バト,Engineer2
13,企書部,都市計書課,NaN,店的六七,円」,Engineer2
14,企書部,都市計書課,NaN,橋架戳右衛門,奉,Engineer2
15,企書部,都市計書課,NaN,村,省下,Engineer2
16,企書部,都市計書課,NaN,武雄五下原田,五下,Engineer2
17,企書部,都市計書課,NaN,日女子立,,Engineer2
18,企書部,都市計書課,NaN,鬻三男,NA,Engineer2


In [71]:
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
with open(path+"Temp.jpg",'rb') as f:
    file_data = f.read()
Json=Clova(file_data)

In [73]:
Json['fields'][]

[{'valueType': 'ALL',
  'boundingPoly': {'vertices': [{'x': 0.0, 'y': 83.0},
    {'x': 10.0, 'y': 83.0},
    {'x': 10.0, 'y': 92.0},
    {'x': 0.0, 'y': 92.0}]},
  'inferText': '山',
  'inferConfidence': 0.9988,
  'type': 'NORMAL',
  'lineBreak': True},
 {'valueType': 'ALL',
  'boundingPoly': {'vertices': [{'x': 0.0, 'y': 100.0},
    {'x': 11.0, 'y': 100.0},
    {'x': 11.0, 'y': 114.0},
    {'x': 0.0, 'y': 114.0}]},
  'inferText': '本',
  'inferConfidence': 0.9993,
  'type': 'NORMAL',
  'lineBreak': True},
 {'valueType': 'ALL',
  'boundingPoly': {'vertices': [{'x': 0.0, 'y': 124.0},
    {'x': 12.0, 'y': 124.0},
    {'x': 12.0, 'y': 138.0},
    {'x': 0.0, 'y': 138.0}]},
  'inferText': '幸',
  'inferConfidence': 0.9983,
  'type': 'NORMAL',
  'lineBreak': True},
 {'valueType': 'ALL',
  'boundingPoly': {'vertices': [{'x': 0.0, 'y': 142.0},
    {'x': 12.0, 'y': 142.0},
    {'x': 12.0, 'y': 156.0},
    {'x': 0.0, 'y': 156.0}]},
  'inferText': '助',
  'inferConfidence': 0.999,
  'type': 'NORMAL',

In [12]:
FailedList

[[34, '企画局', '財務課', '調査掛', 'Worker'],
 [35, '企画局', '財務課', '調査掛', 'Worker'],
 [36, '企画局', '財務課', '調査掛', 'Worker'],
 [37, '企画局', '財務課', '調査掛', 'Worker'],
 [38, '企画局', '財務課', '調査掛', 'Worker'],
 [39, '企画局', '統計課', '統計掛', 'Manager'],
 [39, '企画局', '統計課', '統計掛', 'Clerk1'],
 [40, '企画局', '統計課', '統計掛', 'Manager'],
 [40, '企画局', '統計課', '統計掛', 'Clerk1'],
 [47, '企画局', '統計課', '調査掛', 'Worker'],
 [48, '企画局', '統計課', '調査掛', 'Worker'],
 [49, '企画局', '統計課', '調査掛', 'Worker'],
 [50, '企画局', '統計課', '人口統計掛', 'Worker'],
 [51, '企画局', '統計課', '人口統計掛', 'Worker'],
 [52, '企画局', '統計課', '人口統計掛', 'Worker'],
 [80, '経理局', '会計課', '收入掛', 'Manager'],
 [81, '経理局', '会計課', '收入掛', 'Manager'],
 [92, '経理局', '主税課', '庶務掛', 'Clerk1'],
 [93, '経理局', '主税課', '庶務掛', 'Clerk1'],
 [94, '経理局', '主税課', '庶務掛', 'Clerk1'],
 [130, '経理局', '用度課', '庶務掛', 'Manager'],
 [130, '経理局', '用度課', '庶務掛', 'Worker'],
 [131, '経理局', '用度課', '庶務掛', 'Manager'],
 [131, '経理局', '用度課', '庶務掛', 'Worker'],
 [132, '経理局', '用度課', '庶務掛', 'Manager'],
 [132, '経理局', '用度課', '庶務掛', 'Wor

In [14]:
FailRate=len(FailedList)/len(df)
if len(FailedList)/len(df)<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList)/len(df))))
else:
    print('残念...Success Rate is '+str(1-(len(FailedList)/len(df))))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Data'] = 1-FailRate
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')

残念...Success Rate is 0.6199575371549894


In [66]:
for n in range(0,len(df)):
    Check(df,n)

[0,"Admin","人事課","庶務掛","Manager"]
